# 🧩 Lösung — Objektorientierte Programmierung: Kundenverwaltung

**Kapitel 2 · Python-Vertiefung**

Diese Lösung implementiert eine `Kunde`-Klasse: automatische Kundennummer-Vergabe, Schadenshistorie, Auffälligkeits-Prüfung, ein Bulk-Import aus `bestandskunden.csv`, und eine lesbare Textdarstellung. Am Ende steht ein kurzes Testskript, das alle geforderten Sachverhalte zeigt.

Die Aufgabenstellung hat bewusst keine Methodennamen vorgegeben — das ist hier nur **eine** mögliche Umsetzung, keine einzig richtige.

## Die Klasse `Kunde`

**Design-Entscheidungen:**
- Die fortlaufende Kundennummer wird über ein **Klassenattribut** (`_naechste_kundennummer`) verwaltet — es gehört zur Klasse selbst, nicht zu einem einzelnen Objekt, und wird bei jeder Neuanlage um 1 erhöht. Das erfüllt die Vorgabe, keine globale Variable außerhalb der Klasse zu benutzen.
- Alle Attribute sind privat (führender Unterstrich). Nachname/Vorname bekommen `@property`-Zugriffe (lesend und schreibend), die Kundennummer nur einen lesenden `@property` — es gibt bewusst **keinen Setter** dafür, das erzwingt die Unveränderlichkeit nach der Anlage.
- Der CSV-Import ist als `@classmethod` gelöst: Sie bekommt `cls` (die Klasse selbst) statt `self` (ein fertiges Objekt), weil sie die Objekte erst *baut* — genau wie in den Folien zu Kapitel 2 gezeigt (dort am Beispiel `aus_csv_zeile`).

In [1]:
class Kunde:
    """Ein Kunde des Kfz-Versicherers mit Schadenshistorie."""

    # Klassenattribut: gehört der Klasse, nicht einem einzelnen Objekt.
    # Wird bei jeder Neuanlage hochgezaehlt -> automatische, eindeutige Nummerierung.
    _naechste_kundennummer = 1000

    def __init__(self, nachname, vorname):
        self._kundennummer = Kunde._naechste_kundennummer
        Kunde._naechste_kundennummer += 1

        self._nachname = nachname
        self._vorname = vorname
        self._schadenshistorie = []

    # --- Properties -----------------------------------------------------

    @property
    def kundennummer(self):
        """Nur lesbar -- kein Setter, damit sie nach der Anlage fix bleibt."""
        return self._kundennummer

    @property
    def nachname(self):
        return self._nachname

    @nachname.setter
    def nachname(self, wert):
        self._nachname = wert

    @property
    def vorname(self):
        return self._vorname

    @vorname.setter
    def vorname(self, wert):
        self._vorname = wert

    # --- Schadensverwaltung ----------------------------------------------

    def schaden_melden(self, betrag):
        """Fuegt einen neuen Schaden (in EUR) zur Historie hinzu."""
        self._schadenshistorie.append(betrag)

    def schaden_entfernen(self):
        """Bonus: entfernt den zuletzt gemeldeten Schaden wieder (z.B. bei
        einer irrtuemlichen Meldung) und gibt ihn zurueck."""
        if not self._schadenshistorie:
            return None
        return self._schadenshistorie.pop()

    def durchschnittlicher_schaden(self):
        """Mittelwert der Schadenshistorie, 0 falls noch kein Schaden gemeldet."""
        if not self._schadenshistorie:
            return 0
        return sum(self._schadenshistorie) / len(self._schadenshistorie)

    def ist_auffaellig(self, schwelle=2000):
        """True, wenn der durchschnittliche Schaden über der Schwelle liegt."""
        return self.durchschnittlicher_schaden() > schwelle

    # --- Bulk-Import aus CSV ----------------------------------------------

    @classmethod
    def aus_csv_datei(cls, pfad):
        """Liest bestandskunden.csv ein und gibt eine Liste neuer Kunde-Objekte
        zurueck -- ein Objekt pro Datenzeile (die Kopfzeile wird ueberlesen)."""
        kunden = []
        with open(pfad, encoding="utf-8") as datei:
            zeilen = datei.readlines()

        for zeile in zeilen[1:]:  # erste Zeile ist die Kopfzeile "Nachname,Vorname"
            zeile = zeile.strip()
            if not zeile:
                continue
            nachname, vorname = zeile.split(",")
            kunden.append(cls(nachname.strip(), vorname.strip()))

        return kunden

    # --- Textdarstellung ---------------------------------------------------

    def __str__(self):
        durchschnitt = self.durchschnittlicher_schaden()
        # Deutsches Zahlenformat: Punkt als Tausender-, Komma als Dezimaltrennzeichen
        durchschnitt_de = f"{durchschnitt:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
        auffaellig_text = "Ja" if self.ist_auffaellig() else "Nein"
        return (
            f"Kd.-Nr. {self.kundennummer}: {self.nachname}, {self.vorname}\n"
            f"Anzahl Schäden: {len(self._schadenshistorie)}, "
            f"Ø Schadenshöhe: {durchschnitt_de} EUR\n"
            f"Auffällig: {auffaellig_text}"
        )


## Testskript

Zeigt alle geforderten Sachverhalte:
1. den gesamten Bestand aus `bestandskunden.csv` einlesen, Anzahl ausgeben
2. zeigen, dass die eingelesenen Kunden fortlaufende, eindeutige Kundennummern erhalten
3. für einen Kunden mehrere Schäden melden, Durchschnitt und Auffälligkeit ausgeben

In [2]:
# 1. Gesamten Bestand einlesen
bestand = Kunde.aus_csv_datei("bestandskunden.csv")
print(f"Anzahl eingelesener Kunden: {len(bestand)}")


Anzahl eingelesener Kunden: 10


In [3]:
# 2. Fortlaufende, eindeutige Kundennummern pruefen
nummern = [k.kundennummer for k in bestand]
print("Kundennummern:", nummern)
assert nummern == sorted(set(nummern)), "Nummern sollten aufsteigend und eindeutig sein"


Kundennummern: [1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009]


In [4]:
# 3. Fuer den ersten eingelesenen Kunden mehrere Schäden melden
kunde_a = bestand[0]
kunde_a.schaden_melden(800)
kunde_a.schaden_melden(1100)
kunde_a.schaden_melden(5450)

print(kunde_a)


Kd.-Nr. 1000: Sauerland, Wilfried
Anzahl Schäden: 3, Ø Schadenshöhe: 2.450,00 EUR
Auffällig: Ja


In [5]:
# Kunde ohne gemeldete Schäden -- Kontrollfall
kunde_b = bestand[1]
print(kunde_b)
assert kunde_b.durchschnittlicher_schaden() == 0
assert kunde_b.ist_auffaellig() is False


Kd.-Nr. 1001: Meier, Petra
Anzahl Schäden: 0, Ø Schadenshöhe: 0,00 EUR
Auffällig: Nein


## Bonus: zuletzt gemeldeten Schaden entfernen

In [6]:
entfernt = kunde_a.schaden_entfernen()
print(f"Entfernter Schaden: {entfernt} EUR")
print(kunde_a)


Entfernter Schaden: 5450 EUR
Kd.-Nr. 1000: Sauerland, Wilfried
Anzahl Schäden: 2, Ø Schadenshöhe: 950,00 EUR
Auffällig: Nein


## Kontrolle: Kundennummer ist unveränderlich

Ein Zuweisungsversuch auf `kundennummer` muss fehlschlagen, weil es keinen Setter gibt.

In [7]:
try:
    kunde_a.kundennummer = 9999
except AttributeError as e:
    print("Erwarteter Fehler:", e)


Erwarteter Fehler: property 'kundennummer' of 'Kunde' object has no setter
